Simple chat bot 


Chat bot with summarized memory 
Create a LLM \
Node START \
Node - ChatNode (chats with the users )\
Node - Summizier (summizeries the messagses )\
Node - END\
Edge -  Summizer or END 


STATE(Message state and summary)

In [12]:
#Import all the modules 
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState  , StateGraph , START , END
from langchain.messages import HumanMessage , SystemMessage  , RemoveMessage
from dotenv import load_dotenv


Create a LLM


In [7]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model =  "gemini-3-flash-preview"
)

Define a state 


In [8]:
class State(MessagesState):
    summary :str

Chat functions 


In [ ]:
def call_model(state : State):
    "Returns the llm invoke method"
    #Get the summary is present 
    summary = state.get('summary' , "")
    if summary:
        #add a system prompt 
        summary_prompt =  f"This is the summary up to date : {summary}"
        messages = [SystemMessage(content=summary_prompt)] + state["messages"]
    else :
        messages = state['messages']
    #Invoke the llm model
    response = llm.invoke(messages)
    return response




In [15]:
def summarize_messages(state:State):
    "Summarizes the complete messages"

    summary = state.get("summary" , "")
    #Check weather the summary is present 
    if summary :
        #Add sumamry and the intruction 
        summary_messages = (
            f"This is the summary till date : {summary}"
            "Extend the summary by adding details from the lastest messages"
        ) 
    else :
        summary_messages =  "Create a summary of the lastest messages"
    

    messages = state['messages'] + [HumanMessage(content=summary_messages)]
    #Get the response 
    response = llm.invoke(messages)
    #Delete the messages 
    delete_messages = [RemoveMessage(id=m.id) for m in state['messages'][:-2]]

    return {
        "summary" : response.content , 
        "messages" : delete_messages
    }

In [16]:
def summary_conditional_node(state : State):
    "Checks the conditon to summarize or to end the conversation"

    if len(state['messages']) > 6 :
        return "summarize_messages"
    return END